# An Instructional Guide for Implementing a Low-resource Decentralized Semantic Search Data Catalog

## Context

### Motivation  

Organizations with limited computing resources often face challenges in making their collections of reports, surveys, and indicators easily discoverable. Traditional keyword-based search methods, while widely used, frequently fail to retrieve relevant results when users phrase queries differently from how the data is indexed. This problem is especially pronounced in fields with specialized terminology, where strict keyword matching may fail to recognize conceptually similar terms used in different contexts. For example, in socioeconomic data, a user searching for _"child hunger rates"_ may not retrieve relevant reports labeled under _"child malnutrition prevalence"_ or _"undernutrition in children under five"_, even though they describe closely related concepts.

**Semantic search** offers a significant improvement by retrieving results based on meaning rather than exact keyword matches. However, most implementations require specialized hardware, backend infrastructure for embedding generation, and a vector database for efficient retrieval—all of which can be prohibitively expensive or technically complex for resource-constrained organizations.

**Low-resource AI models** provide a practical pathway for organizations to adopt semantic search without the need for resource-intensive operational systems. Unlike traditional deep learning models that rely on high-performance GPUs, cloud-based inference, or dedicated vector databases, these models are optimized for efficiency, allowing them to run on consumer-grade hardware, edge devices, or directly in web browsers.

By leveraging **lighter architectures**, **quantization**, **distillation**, or **client-side execution**, low-resource AI models can maintain strong performance while significantly reducing computational demands. This eliminates the need for backend infrastructure and GPU-powered servers, enabling organizations to deploy cost-effective, scalable, and privacy-preserving AI solutions without managing complex machine learning pipelines.

This notebook presents a **low-resource semantic search** approach tailored for NSOs, development organizations, and other institutions managing a few thousand resources, where enhancing data discoverability is essential but backend infrastructure is limited or cost-prohibitive.

By running the entire search pipeline entirely in the browser using JavaScript-based embeddings and similarity search, this solution removes the need for a dedicated vector database and reduces reliance on costly cloud computing services—**making AI-powered semantic search accessible even in low-resource settings**.

<!-- https://www.semrush.com/blog/semantic-search/ -->
<!-- ![Semantic search - Semrush](https://static.semrush.com/blog/uploads/media/31/50/3150dd9c369ec2c272658bdfb161ad3f/d78b61c1a1a21dac3d16147e9cb4852e/FLhzdoFxIHH23S-htv5mDsTi-bzzpric_UPiNDG8EH8TZvqEqv3FaXNVA4dNjMzXTK09stsR7mGjTH4TfJAEfPZN_KE91ZUND-6swWj9VFhtdMPNAyyFHq9sSdvxiBvHzhNFnExJBVVL5ZXupob8Cpc.png) -->


### Background  

Most semantic search implementations rely on **vector databases** (e.g., FAISS, Qdrant) to store and retrieve embeddings. This setup, while effective, requires both **server-side processing** for embedding generation and **specialized indexing** to enable efficient retrieval. In some cases, this complexity may not be necessary especially when the data is only a few hundreds or thousands and the use case only requires one vector embedding for each item in the data.

Instead, this **low-resource alternative** operates as follows:

- **Client-side inference with `transformers.js`**  
  - A pre-trained transformer-based embedding model runs **directly in the browser**, generating query embeddings without requiring a server.
  - This approach removes the need for API calls to an external embedding service, making it cost-efficient and privacy-preserving.

- **JavaScript-based similarity search**  
  - Instead of querying a vector database, embeddings are stored in-memory and searched using **cosine similarity**.  

- **Offline generation of data embeddings**
  - The generation of embeddings for data in the catalog is generated offline and stored as JSON data.

### Why Now?  

With the advancement of **on-device AI inference** and **optimized JavaScript implementations of transformers**, this paradigm shift is now viable. Previously, running transformer-based models on the client side was impractical due to hardware limitations. However, modern web browsers and devices are now capable of efficiently performing **small-scale deep learning inference**, enabling a new class of **decentralized AI applications**. Additionally, efforts towards **developing high-performance low-resource AI models** are accelerating, making this paradigm of productionizing AI applications more appealing.

By adopting this low-resource approach, NSOs and organizations with limited compute resources can deploy **affordable, scalable, and privacy-preserving** semantic search solutions. This enables more effective dissemination of data and helps unlock the value of statistical information for policymakers, researchers, and the public.

# Preview

By the end of this guide, you'll have your own data catalog—just like the one below—featuring semantic search that runs entirely in the user's browser.

Try entering complex or natural language queries to see semantic search in action. The system will attempt to rank all items in the catalog based on their relevance to your query.

**NOTE:** Even if no truly relevant data exists, the system will still return the most semantically similar results it can find.

<script type="text/javascript" src="https://pym.nprapps.org/pym.v1.min.js"></script>

In [ ]:
from IPython.display import HTML, IFrame
IFrame(
    src="https://avsolatorio.github.io/ai-for-data-blog/semantic-search/test-index.html",
    width="100%",
    scrolling="yes",
    marginheight="0",
    frameborder="0",
    height="800px"
)

# Technical guide

## Contents

---

The technical guide is divided into the following parts.

The first part will cover collecting metadata from a catalog. In this case, we will show the WDI as an example.

The second part is identifying an open source embedding model and creating an ONNX compliant version (if not yet available). Then, we will show how to create embeddings from the metadata collected from the previous step.

Next, we will show how to configure GitHub to host the files.


The following sections will guide you through implementing this system, covering:

1. Setting up `transformers.js` for client-side embedding generation.
2. Storing and retrieving document embeddings without a vector database.
3. Implementing an efficient similarity search function in JavaScript.
4. Optimizing performance for low-resource environments.

## JavaScript packages and setup

While the javascript packages below are available on this repository, we are showing the contents and the process of compiling these files into a bundle so you can later costumize these as needed.

First, we will install the necessary packages and dependencies.

In [ ]:
%%capture
%%bash

# Update and install npm
apt update && apt install npm

# Install browserify
sudo npm install browserify -g

# Define the JS project directory
PROJ_DIR=/content/js

# Create the PROJ_DIR if it doesn't exist
if [ ! -d $PROJ_DIR ]; then
    mkdir -p $PROJ_DIR
fi

# Create the directory for the search files
mkdir -p $PROJ_DIR/search

# Change to the PROJ_DIR
cd $PROJ_DIR


# Install the BM25 search and cosine similarity libraries
npm install wink-bm25-text-search compute-cosine-similarity --save

# Install esbuild for building the package
npm install --save-dev esbuild

# # Create a bundle
# browserify -r wink-bm25-text-search:bm25 -r compute-cosine-similarity:similarity > ss-bundle.js

We create a package that contains both BM25 and cosine similarity libraries. We separate the files for semantic search and keyword search for modularity, but we also create a bundle that contains both. We write these to the `js/search/semantic.js`, `js/search/keyword.js`, and `js/search/search.js` files.

In [ ]:
%%writefile js/search/semantic.js
// Semantic search powered by Transformers from Hugging Face
import { pipeline } from 'https://cdn.jsdelivr.net/npm/@xenova/transformers';
import similarity from 'compute-cosine-similarity';

let extractor = null;

/**
 * Loads the HuggingFace embedding pipeline.
 * @param {string} organization - HF model org, e.g., "avsolatorio"
 * @param {string} modelName - Model name, e.g., "GIST-all-MiniLM-L6-v2"
 * @returns {Promise<Object>}
 */
export async function loadExtractor(organization = 'avsolatorio', modelName = 'GIST-all-MiniLM-L6-v2') {
    if (!extractor) {
        extractor = await pipeline("feature-extraction", `${organization}/${modelName}`);
    }
    return extractor;
}

/**
 * Returns the sentence embedding vector for input text.
 * @param {string} text
 * @param {string} organization
 * @param {string} modelName
 * @returns {Promise<Array<number>>}
 */
export async function getEmbedding(text, organization = 'avsolatorio', modelName = 'GIST-all-MiniLM-L6-v2') {
    const model = await loadExtractor(organization, modelName);
    const result = await model.model(model.tokenizer(text));
    return result.sentence_embedding.data;
}

/**
 * Computes dot product between two equal-length vectors.
 * @param {Array<number>} a
 * @param {Array<number>} b
 * @returns {number}
 */
export function dotProduct(a, b) {
    return a.map((x, i) => x * b[i]).reduce((sum, val) => sum + val, 0);
}

// Optional export
export { similarity };

Writing js/search/semantic.js


In [ ]:
%%writefile js/search/keyword.js
// wink-bm25-text-search provides a BM25-based full-text search engine.
import bm25 from 'wink-bm25-text-search';
import winkNLP from 'wink-nlp';
import winkModel from 'wink-eng-lite-web-model';

// Load the NLP model
const nlp = winkNLP(winkModel);
const its = nlp.its;

/**
 * Prepares a text for BM25 search by:
 * - Tokenizing
 * - Filtering to only include non-stopwords
 * - Adding negation markers
 * - Stemming words
 * @param {string} text
 * @returns {Array<string>} Array of processed tokens
 */
export function prepTask(text) {
    const tokens = [];
    nlp.readDoc(text)
        .tokens()
        .filter((t) => (t.out(its.type) === 'word' && !t.out(its.stopWordFlag)))
        .each((t) => {
            const token = t.out(its.negationFlag) ? '!' + t.out(its.stem) : t.out(its.stem);
            tokens.push(token);
        });
    return tokens;
}

/**
 * Creates and initializes a BM25 engine from data using wink-bm25-text-search.
 * @param {Array<Object>} data - The documents to index.
 * @param {Object} fldWeights - Field weightings for BM25 scoring.
 * @returns {Object} Configured BM25 engine.
 */
export function createBM25Engine(data, fldWeights = { name: 2, definition: 1 }) {
    const engine = bm25();
    engine.defineConfig({ fldWeights });
    engine.definePrepTasks([prepTask]);

    data.forEach((doc, i) => engine.addDoc(doc, i));
    engine.consolidate();

    return engine;
}

// Export for reference or re-use
export { bm25, winkNLP, winkModel, nlp };

Writing js/search/keyword.js


In [ ]:
%%writefile js/search/search.js
export * from './keyword.js';
export * from './semantic.js';

Overwriting js/search/search.js


We also write a utility library that contains helper functions.

In [ ]:
%%writefile js/search/utils.js

/**
 * Normalizes a search query by trimming and converting to lowercase.
 * Useful for consistent comparisons and filtering.
 * @param {string} query
 * @returns {string}
 */
export function normalizeQuery(query) {
    return query.trim().toLowerCase();
}


/**
 * Highlights matching terms in a string by wrapping them in <strong> tags.
 * @param {string} text - The original text.
 * @param {string} query - The term to highlight.
 * @returns {string} - The HTML-formatted string with highlights.
 */
export function highlightMatches(text, query) {
    const matchExists = text.toLowerCase().includes(normalizeQuery(query));
    if (!matchExists) return text;

    const re = new RegExp(query, "ig");
    return text.replace(re, (matchedText) => `<strong>${matchedText}</strong>`);
}

/**
 * Fetches JSON data from a given URL.
 * @param {string} dataUrl - The URL to fetch JSON data from.
 * @returns {Promise<any>} - The parsed JSON data, or null on failure.
 */
export async function dataFetch(dataUrl) {
    try {
        const response = await fetch(dataUrl);
        if (!response.ok) throw new Error(`Failed to fetch: ${dataUrl}`);
        const data = await response.json();
        return data;
    } catch (error) {
        console.error("dataFetch error:", error);
        return null;
    }
}

/**
 * Simulates an API response by returning a slice of data with an optional delay.
 * @param {Array} data - The full dataset.
 * @param {Array} buffer - The currently loaded subset of data.
 * @param {number} topN - The number of new items to return.
 * @param {number} delay - The artificial delay in milliseconds.
 * @returns {Promise<Array>} - The new batch of items.
 */
export async function mockApi(data, buffer, topN = 10, delay = 500) {
    return new Promise(resolve => {
        setTimeout(() => {
            const res = data.slice(buffer.length, buffer.length + topN);
            resolve(res);
        }, delay);
    });
}

/**
 * Handles paginated loading of additional data with delay simulation.
 * @param {Object} options
 * @param {Array} options.fullData - The entire dataset.
 * @param {Array} options.currentData - The current buffer (already loaded).
 * @param {number} options.topN - The batch size to fetch.
 * @param {number} options.delay - Optional delay in milliseconds.
 * @returns {Promise<Object>} - `{ status: 'ok'|'empty', data: [] }`
 */
export async function paginatedLoad({ fullData, currentData, topN = 10, delay = 500 }) {
    if (currentData.length >= fullData.length) {
        return { status: 'empty', data: [] };
    } else {
        const data = await mockApi(fullData, currentData, topN, delay);
        return { status: 'ok', data };
    }
}

Overwriting js/search/utils.js


After the files are written, we should build and minify them so they become smaller in size. This will make loading the scripts faster and consume lower bandwith, ideal for low-resource access.

In [ ]:
%%bash

SEARCH_PROJ_DIR=/content/js/search

for FILENAME in utils semantic keyword search; do
  npx esbuild "${SEARCH_PROJ_DIR}/${FILENAME}.js" \
    --bundle --minify \
    --outfile="${SEARCH_PROJ_DIR}/${FILENAME}.min.js"
done


  js/search/utils.min.js  635b 

⚡ Done in 9ms

  js/search/semantic.min.js  3.3kb

⚡ Done in 12ms

  js/search/keyword.min.js  3.5mb ⚠️

⚡ Done in 454ms

  js/search/search.min.js  3.5mb ⚠️

⚡ Done in 463ms


After generating these files, you can upload these to your hosting server or on GitHub with pages activated.

# Building a Decentralized Semantic Search Data Catalog

Now that the requisite javascript files are available, we can demonstrate how to create a semantic search data catalog that's fully decentralized and only leverage GitHub pages for hosting the needed static content, e.g., HTML, JavaScript, metadata, and pre-computed embeddings.

First, let's collect metadata from some catalog. Then, we write Python scripts that will process the metadata and generate the semantic embeddings. We will then upload the embeddings.

## Formatting the metadata and generating indicators

Here we use the World Development Indicators (WDI) and the Policy Research Working Papers (PRWP) data to test how this semantic search system can power the discoverability of a few thousand resources.

## Generating embeddings

After collecting the metadata of the samples we want to put in the catalog, we will need to generate the embeddings for the texts that we want to use as basis for finding the data.

The embedding generation requires selecting an embedding model that will be used to convert the text metadata into numerical representations that encode semantic meaning.

A list of embedding models can be found in: https://huggingface.co/spaces/mteb/leaderboard.

For this, we will use the [`avsolatorio/GIST-all-MiniLM-L6-v2`](https://huggingface.co/avsolatorio/GIST-all-MiniLM-L6-v2) which is a lightweight embedding model that can run on mobile devices.

## Indicators

In [ ]:
!pip install fire hf_xet &> /dev/null
!apt install brotli &> /dev/null

In [ ]:
import os
import json
import pandas as pd
# import fire
from sentence_transformers import SentenceTransformer
from typing import Optional

PREVIEW_FIELDS = ["idno", "title", "text", "type", "type_extra", "source", "time_coverage", "geographic_coverage"]


def get_time_coverage(df):
    t_range = df[df.columns[4:]].dropna(axis=0, thresh=1).sum(axis=0)
    t_range = t_range[t_range > 0]
    if t_range.empty:
        return None

    t_range = t_range.sort_index()

    if t_range.index[0] == t_range.index[-1]:
        return t_range.index[0]

    return f"{t_range.index[0]} - {t_range.index[-1]}"

def get_geographic_coverage(df):
    ddf = df[df.columns[4:]].dropna(axis=1, thresh=1)
    if ddf.empty:
        return None

    df = df.loc[ddf.index]

    return df["Country Name"].tolist()


def get_data_coverage(df_data: pd.DataFrame) -> pd.DataFrame:
    tc = df_data.groupby("Indicator Code").apply(get_time_coverage, include_groups=False)
    tc = tc.reset_index()
    tc.columns = ["idno", "time_coverage"]

    gc = df_data.groupby("Indicator Code").apply(get_geographic_coverage, include_groups=False)
    gc = gc.reset_index()
    gc.columns = ["idno", "geographic_coverage"]

    df = pd.merge(tc, gc, on="idno")

    return df

def get_preview_metadata(df_metadata: pd.DataFrame, df_data: pd.DataFrame | None = None) -> pd.DataFrame:
    df = df_metadata.rename(columns={
        "Series Code": "idno",
        "Indicator Name": "title",
    })

    # Derive text from Long definition if present, or Short definition
    df["text"] = df["Long definition"].fillna(df["Short definition"])
    df["type"] = "indicator"
    df["type_extra"] = "World Development Indicators (WDI)"
    df["source"] = "World Bank, Development Data Group"

    if df_data is not None:
        df = pd.merge(df, get_data_coverage(df_data), on="idno")

    return df[[i for i in PREVIEW_FIELDS if i in df.columns]]

# 0
# :
# {metadata_type: "indicator", type: "indicator", idno: "SI.POV.GAPS",…}
# geographic_coverage
# :
# ["Albania", "Algeria", "Angola", "Argentina", "Armenia", "Australia", "Austria", "Azerbaijan",…]
# idno
# :
# "SI.POV.GAPS"
# metadata_type
# :
# "indicator"
# source
# :
# ["World Bank, Development Data Group"]
# sub_title
# :
# null
# text
# :
# "Poverty gap at $2.15 a day (2017 PPP) is the mean shortfall in income or consumption from the poverty line $2.15 a day (counting the nonpoor as having zero shortfall), expressed as a percentage of the poverty line. This measure reflects the depth of poverty as well as its incidence."
# thumbnail
# :
# null
# time_coverage
# :
# "1963 - 2023"
# title
# :
# "Poverty gap at $2.15 a day (2017 PPP) (%)"
# type
# :
# "indicator"
# type_extra
# :
# "World Development Indicators (WDI)"

#     # export interface Metadata extends Item {
#     # id?: number
#     # idno?: string | number
#     # source?: string[]
#     # geographic_coverage?: string[]
#     # geographic_coverage_title?: string
#     # defaultBilling?: boolean
#     # sub_title?: string
#     # text?: string
#     # thumbnail?: string
#     # time_coverage?: string
#     # title?: string
#     # type?: string
#     # type_extra?: any
#     # abstract?: string
#     # dimensions?: string[]
#     # collection?: string
#     # doi?: string
#     # }


# def load_indicator_data(
#     file_path: str,
#     name_col: str = "Indicator Name",
#     definition_col: str = "Long definition",
#     fallback_definition_col: Optional[str] = "Short definition",
#     code_col: str = "Series Code",
#     sheet_name: str = "Series"
# ) -> pd.DataFrame:
#     """
#     Load and prepare indicator data from an Excel or CSV file.

#     Parameters
#     ----------
#     file_path : str
#         Path to the Excel or CSV file containing indicator metadata.
#     name_col : str
#         Column name containing the indicator name.
#     definition_col : str
#         Primary column for definitions.
#     fallback_definition_col : Optional[str]
#         Backup column to use if primary definition is missing.
#     code_col : str
#         Column with the unique code identifier for each indicator.

#     Returns
#     -------
#     pd.DataFrame
#         DataFrame with columns: ['code', 'name', 'definition'].
#     """
#     # Load Excel or CSV
#     ext = os.path.splitext(file_path)[-1].lower()
#     if ext == '.xlsx':
#         df = pd.read_excel(file_path, sheet_name=sheet_name or 0)
#     elif ext == '.csv':
#         df = pd.read_csv(file_path)
#     else:
#         raise ValueError(f"Unsupported file type: {ext}")

#     # Prepare columns
#     df["definition"] = df[definition_col]
#     if fallback_definition_col:
#         missing_def = df["definition"].isnull()
#         df.loc[missing_def, "definition"] = df.loc[missing_def, fallback_definition_col]

#     df.rename(columns={
#         code_col: "code",
#         name_col: "name"
#     }, inplace=True)

#     return df[["code", "name", "definition"]]


def generate_embeddings(
    df: pd.DataFrame,
    model_name: str = "avsolatorio/GIST-all-MiniLM-L6-v2",
    main_text_col: str = "text",
) -> pd.DataFrame:
    """
    Generate sentence embeddings from a dataframe containing text data.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with columns ['code', 'name', 'definition'].
    model_name : str
        Hugging Face model path or name for SentenceTransformer.

    Returns
    -------
    pd.DataFrame
        DataFrame with added 'embedding' column.
    """
    model = SentenceTransformer(model_name)
    # text_input = df["name"] + "\n\n" + df["definition"]
    text_input = df["title"] + "\n\n" + df[main_text_col]
    df["embedding"] = model.encode(text_input, show_progress_bar=True).tolist()
    return df


def save_outputs(
    df: pd.DataFrame,
    output_dir: str,
    base_filename: str,
    dtype: str,
    double_precision: int = 10,
    preview_fields: list[str] | None = None,
):
    """
    Save processed definitions and embeddings to JSON files.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with columns ['code', 'name', 'definition', 'embedding'].
    output_dir : str
        Directory to save the output files.
    base_filename : str
        Base filename prefix for saving the outputs.
    double_precision : int
        Number of decimal places for embedding values.
    """
    os.makedirs(output_dir, exist_ok=True)

    # definitions_path = os.path.join(output_dir, "definitions.json")
    embeddings_path = os.path.join(output_dir, f"{base_filename}__{double_precision or 0:03}__{dtype}_embeddings.json")

    # df.set_index("code")["definition"].to_json(definitions_path)
    # df[["code", "name", "definition", "embedding"]].to_json(embeddings_path, orient="records", double_precision=double_precision)

    if preview_fields is None:
        preview_fields = df.columns

    df[[i for i in preview_fields if i in df.columns and i != "embedding"] + ["embedding"]].to_json(embeddings_path, orient="records", double_precision=double_precision)


def main(
    # input_file: str = "./data/WDIEXCEL.xlsx",
    df_metadata: pd.DataFrame = None,
    df_data: pd.DataFrame = None,
    output_dir: str = "./data",
    model_name: str = "avsolatorio/GIST-all-MiniLM-L6-v2",
    double_precision: int = 10,
    main_text_col: str = "text",
    preview_fields: list[str] | None = None,
    dtype: str = "indicator",
):
    """
    Main function to load indicator metadata, generate embeddings, and save the results.

    Parameters
    ----------
    input_file : str
        Path to the input Excel or CSV file.
    output_dir : str
        Directory where output JSON files will be saved.
    model_name : str
        SentenceTransformer model to use for embeddings.
    double_precision : int
        Number of decimal places for embedding values.
    """
    base_model = model_name.replace("/", "__")  # For output file naming
    # df = load_indicator_data(input_file)

    df = get_preview_metadata(df_metadata, df_data)
    df = generate_embeddings(df, model_name, main_text_col=main_text_col)
    save_outputs(df, output_dir, base_model, dtype, double_precision=double_precision, preview_fields=preview_fields)

    return df

# if __name__ == "__main__":
#     # Usage:
#     # poetry run python generate_embeddings.py --input_file="../data/WDIEXCEL.xlsx"
#     fire.Fire(main)

In [ ]:
!unzip WDI_EXCEL\ -\ 20250519.zip

Archive:  WDI_EXCEL - 20250519.zip
  inflating: WDIEXCEL.xlsx           


In [ ]:
## Set up files

input_file = "./data/WDIEXCEL.xlsx"
output_dir = "./data"
# model_name = "avsolatorio/GIST-all-MiniLM-L6-v2"
model_name = "avsolatorio/GIST-small-Embedding-v0"
base_model = model_name.replace("/", "__")  # For output file naming
double_precision = 5

df_metadata = pd.read_excel("./WDIEXCEL.xlsx", sheet_name="Series")
df_data = pd.read_excel("./WDIEXCEL.xlsx", sheet_name="Data")

df = main(df_metadata=df_metadata, df_data=df_data, output_dir=output_dir, model_name=model_name, double_precision=double_precision, preview_fields=PREVIEW_FIELDS)
df

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/68.0k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/48 [00:00<?, ?it/s]

,idno,title,text,type,type_extra,source,time_coverage,geographic_coverage,embedding
0,AG.CON.FERT.PT.ZS,Fertilizer consumption (% of fertilizer produc...,Fertilizer consumption measures the quantity o...,indicator,World Development Indicators (WDI),"World Bank, Development Data Group",1961 - 2022,"[Africa Eastern and Southern, Africa Western a...","[0.02338995784521103, -0.007264019455760717, 0..."
1,AG.CON.FERT.ZS,Fertilizer consumption (kilograms per hectare ...,Fertilizer consumption measures the quantity o...,indicator,World Development Indicators (WDI),"World Bank, Development Data Group",1961 - 2022,"[Africa Eastern and Southern, Africa Western a...","[0.01732984185218811, -0.004328237380832434, 0..."
2,AG.LND.AGRI.K2,Agricultural land (sq. km),Agricultural land refers to the share of land ...,indicator,World Development Indicators (WDI),"World Bank, Development Data Group",1961 - 2021,"[Africa Eastern and Southern, Africa Western a...","[0.044430673122406006, -0.05081179738044739, 0..."
3,AG.LND.AGRI.ZS,Agricultural land (% of land area),Agricultural land refers to the share of land ...,indicator,World Development Indicators (WDI),"World Bank, Development Data Group",1961 - 2022,"[Africa Eastern and Southern, Africa Western a...","[0.04052712023258209, -0.04145373776555061, 0...."
4,AG.LND.ARBL.HA,Arable land (hectares),Arable land (in hectares) includes land define...,indicator,World Development Indicators (WDI),"World Bank, Development Data Group",1961 - 2021,"[Africa Eastern and Southern, Africa Western a...","[0.019351061433553696, -0.02011195383965969, 0..."
...,...,...,...,...,...,...,...,...,...
1504,VC.IDP.NWCV,"Internally displaced persons, new displacement...",Internally displaced persons are defined accor...,indicator,World Development Indicators (WDI),"World Bank, Development Data Group",2009 - 2023,"[Africa Eastern and Southern, Africa Western a...","[-0.0281879473477602, -0.021369975060224533, 0..."
1505,VC.IDP.NWDS,"Internally displaced persons, new displacement...",Internally displaced persons are defined accor...,indicator,World Development Indicators (WDI),"World Bank, Development Data Group",2008 - 2023,"[Africa Eastern and Southern, Africa Western a...","[-0.022089587524533272, -0.02350389026105404, ..."
1506,VC.IHR.PSRC.FE.P5,"Intentional homicides, female (per 100,000 fem...","Intentional homicides, female are estimates of...",indicator,World Development Indicators (WDI),"World Bank, Development Data Group",1990 - 2021,"[Africa Eastern and Southern, Africa Western a...","[-0.033108778297901154, -0.033592816442251205,..."
1507,VC.IHR.PSRC.MA.P5,"Intentional homicides, male (per 100,000 male)","Intentional homicides, male are estimates of u...",indicator,World Development Indicators (WDI),"World Bank, Development Data Group",1990 - 2021,"[Africa Eastern and Southern, Africa Western a...","[-0.009725609794259071, -0.005280175246298313,..."


In [ ]:
df = truncated_df = pd.read_json("/content/data/avsolatorio__GIST-all-MiniLM-L6-v2__005__embeddings.json", orient="records")

In [ ]:
# Compare the loss incurred by truncation
from sklearn.metrics.pairwise import cosine_similarity

full_df = pd.read_json(f"./data/{base_model}__010__embeddings.json", orient="records")
truncated_df = pd.read_json(f"./data/{base_model}__005_embeddings.json", orient="records")

sims = cosine_similarity(np.vstack(full_df["embedding"]), np.vstack(truncated_df["embedding"]))
is_close = np.isclose(sims.diagonal(), 1.0, atol=1e-5)  # or use rtol for relative tolerance

assert is_close.all()

## Documents

In [ ]:
# https://search.worldbank.org/api/v2/wds?format=json&fct=docty_exact,count_exact,lang_exact,disclstat_exact&order=desc&majdocty_key=658102&rows=20&apilang=en&os=0&srt=docdt&projectid=P006577
import requests
import os
import re
from tqdm.auto import tqdm


# Define the base URL
BASE_URL = "https://search.worldbank.org/api/v2/wds"


def get_content(metadata):
    title = re.sub(r'\s+', ' ', metadata["display_title"])
    content = title + "\n\n"

    if "abstracts" in metadata:
        abstract = metadata["abstracts"]['cdata!']
        content += re.sub(r'\s+', ' ', abstract)

    return content.strip()


def fetch_prwp_metadata(os: int = 0, rows: int = 1000, order: str = "asc", get_total: bool = False):
    assert order in ("asc", "desc")

    # Define the query parameters as a dictionary
    params = {
        "format": "json",
        # "fct": "docty_exact,count_exact,lang_exact,disclstat_exact",
        "order": "desc",
        "rows": rows,
        "apilang": "en",
        "os": os,
        "srt": "docdt",
        "order": order,
        "lang_exact": "English",
        "docty_exact": "Policy Research Working Paper"
    }

    # Make the GET request
    response = requests.get(BASE_URL, params=params)

    # Check if the request was successful
    if response.status_code == 200:
        data = response.json()
        # print(data)  # Process the JSON response as needed
    else:
        print(f"Error: {response.status_code}, {response.text}")

    total = data["total"]

    if get_total:
        print(f"Total: {total}")
        return total

    docs = data["documents"]
    docs.pop("facets", None)

    return data["documents"]

In [ ]:
metadata = {}

In [ ]:
total = fetch_prwp_metadata(get_total=True)

os = 0
rows = 1000

while len(metadata) < total:
    out = fetch_prwp_metadata(os=os, rows=rows)
    metadata.update(out)

    os += rows

    if os > total:
        break

In [ ]:
import os
from os import read
import json
import pandas as pd

# Take this from IDS `./details-info/`
doc_df = pd.read_json("./document_details_info.jsonl", lines=True)

In [ ]:
output_dir = "./data"
# model_name = "avsolatorio/GIST-all-MiniLM-L6-v2"
model_name = "avsolatorio/GIST-small-Embedding-v0"
base_model = model_name.replace("/", "__")  # For output file naming
double_precision = 5

orig_doc_emb_df = doc_emb_df = generate_embeddings(doc_df, model_name=model_name, main_text_col="abstract")
doc_emb_df.head()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/68.0k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/342 [00:00<?, ?it/s]

,metadata_type,type,idno,title,thumbnail,type_extra,sub_title,source,geographic_coverage,time_coverage,text,doi,abstract,completeness,embedding
0,document,document,23179104,World development report 1988,NaN,World Development Report,,[World Bank],[World],1988,NaN,,This report is the eleventh in the series of a...,0.846154,"[-0.040269214659929276, 0.001575297093950212, ..."
1,document,document,700163,Restrictive labor practices in seaports,NaN,Policy Research Working Paper,,"[Harding,Alan S.]",None,1990,NaN,,Containerization and modern bulk handling meth...,0.769231,"[-0.00538013968616724, -0.034989144653081894, ..."
2,document,document,23866007,Financial innovation and money demand : theory...,NaN,Policy Research Working Paper,,"[Arrau, Patricio, de Gregorio, Jose]",[World],1991,NaN,,"Empirically, traditional money demand equation...",0.846154,"[0.0023762888740748167, -0.06970712542533875, ..."
3,document,document,697843,A typology of foreign exchange auction markets...,NaN,Policy Research Working Paper,,"[ARON, JANINE, Elbadawi,Ibrahim Ahmed]","[Ghana, Nigeria, Uganda, Zambia]",1994,NaN,,"In this analytical sequel to ""A Typology of Fo...",0.846154,"[-0.03594847768545151, -0.010983501560986042, ..."
4,document,document,698053,Are portfolio flows to emerging markets comple...,NaN,Policy Research Working Paper,,"[Gooptu, Sudarshan]","[Argentina, Brazil, Chile, India, Indonesia, K...",1994,NaN,,Increasing portfolio investment flows to emerg...,0.846154,"[0.006264681462198496, -0.07270757108926773, 0..."


In [ ]:
import numpy as np

def uint8_quantize_embedding(embedding):
    max_abs = np.max(np.abs(embedding))
    scale = max_abs / 127
    quantized = np.round(embedding / scale).astype(np.int8)

    return dict(quantized=quantized, scale=scale)


def uint8_quantize(emb_df, drop_orig: bool = False):
    emb_df = pd.concat([emb_df, emb_df["embedding"].apply(lambda x: pd.Series(uint8_quantize_embedding(x)))], axis=1)

    if drop_orig:
        emb_df.drop(columns=["embedding"], inplace=True)
        emb_df.rename(columns={"quantized": "embedding"}, inplace=True)

    return emb_df

In [ ]:
doc_emb_df = uint8_quantize(orig_doc_emb_df, drop_orig=True)
save_outputs(doc_emb_df, output_dir, base_model, dtype="doc", double_precision=5, preview_fields=None)

In [ ]:
!zip /content/data/avsolatorio__GIST-small-Embedding-v0__005__doc_embeddings.json.zip /content/data/avsolatorio__GIST-small-Embedding-v0__005__doc_embeddings.json
!brotli -k /content/data/avsolatorio__GIST-small-Embedding-v0__005__doc_embeddings.json

  adding: content/data/avsolatorio__GIST-small-Embedding-v0__005__doc_embeddings.json (deflated 66%)


In [ ]:
save_outputs(doc_emb_df, output_dir, base_model, dtype="doc", double_precision=4, preview_fields=None)
save_outputs(doc_emb_df, output_dir, base_model, dtype="doc", double_precision=3, preview_fields=None)

In [ ]:
# Compare the loss incurred by truncation
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

full_df = pd.read_json(f"./data/{base_model}__005__doc_embeddings.json", orient="records")
truncated_df = pd.read_json(f"./data/{base_model}__003__doc_embeddings.json", orient="records")

sims = cosine_similarity(np.vstack(full_df["embedding"]), np.vstack(truncated_df["embedding"]))
is_close = np.isclose(sims.diagonal(), 1.0, atol=1e-5)  # or use rtol for relative tolerance

assert is_close.all()

In [ ]:
!pip install hnswlib nmslib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 kB 16.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached pybind11-2.6.1-py2.py3-none-any.whl.metadata (8.7 kB)
Using cached pybind11-2.6.1-py2.py3-none-any.whl (188 kB)
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for nmslib
  Running setup.py clean for nmslib
Failed to build nmslib
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (nmslib)


In [ ]:
import hnswlib
import numpy as np

# Example: 128-dim vectors, 10,000 points
dim = 128
num_elements = 10000
data = np.random.rand(num_elements, dim).astype(np.float32)

# Initialize HNSW index
p = hnswlib.Index(space='cosine', dim=dim)
p.init_index(max_elements=num_elements, ef_construction=200, M=16)
p.add_items(data)

# This exposes layer info per point
layers = [p.get_layer(point_id) for point_id in range(num_elements)]

# Get neighbors at each level (for a specific point)
neighbors = p.get_connections(point_id)

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 57.1 MB/s eta 0:00:00


In [ ]:
import faiss
import numpy as np

# Example: 10,000 vectors, 512-dim
num_elements = 10000
dim = 512
data = np.random.randn(num_elements, dim).astype(np.float32)

# Initialize HNSW index
M = 16  # Number of neighbors (graph connectivity)
index = faiss.IndexHNSWFlat(dim, M)
index.hnsw.efConstruction = 200  # Construction quality

# Add embeddings to the index
index.add(data)

In [ ]:
# Access neighbors for each node
hnsw = index.hnsw
neighbors = []
for i in range(num_elements):
    neighbor_ids = hnsw.neighbors[i].tolist()
    neighbors.append(neighbor_ids)

# Example: print neighbors for node 0
print("Neighbors of node 0:", neighbors[0])

TypeError: 'Int32Vector' object is not subscriptable

In [ ]:
hnsw.nb_neighbors(0)

32

In [ ]:
neighbors = []
M = index.hnsw.nb_neighbors  # M (number of neighbors per node)

for i in range(num_elements):
    # Slice the neighbors for node i
    neighbor_ids = list(hnsw.neighbors[i * M: (i + 1) * M])
    # Filter out -1 (no neighbor)
    neighbor_ids = [nid for nid in neighbor_ids if nid != -1]
    neighbors.append(neighbor_ids)


In [ ]:
hnsw.neighbors.at((16 * 2) + 1)

5817

In [ ]:
import faiss
import numpy as np
import json
from faiss import vector_to_array  # helper to convert SWIG arrays

# 1) Build the HNSW index
num_elements = 10000
dim = 384
data = np.random.randn(num_elements, dim).astype(np.float32)

M = 16
ef_construction = 200
index = faiss.IndexHNSWFlat(dim, M)
index.hnsw.efConstruction = ef_construction
index.add(data)

hnsw = index.hnsw

# 2) Convert the flat neighbors array into a NumPy array
M0 = hnsw.nb_neighbors(0)                   # slots per node in layer 0
flat_neighbors = vector_to_array(hnsw.neighbors)  # length = num_elements * M0

# 3) Extract per-node neighbor lists
base_neighbors = []
for i in range(num_elements):
    start = i * M0
    nbrs = flat_neighbors[start : start + M0]
    # filter out empty slots (-1)
    nbrs = [int(n) for n in nbrs if n != -1]
    base_neighbors.append(nbrs)

# 4) Convert levels to a NumPy array
levels = vector_to_array(hnsw.levels).astype(int)  # length = num_elements
max_level = int(hnsw.max_level)

# 5) Partition Layer 0 into 100-node shards
shard_size = 100
num_shards = (num_elements + shard_size - 1) // shard_size

for shard_id in range(num_shards):
    start = shard_id * shard_size
    end = min(start + shard_size, num_elements)
    shard = []
    for i in range(start, end):
        shard.append({
            "id": i,
            "vector": data[i].tolist(),
            "neighbors": base_neighbors[i],
            "level": int(levels[i])
        })
    with open(f"layer0_cluster_{shard_id}.json", "w") as f:
        json.dump({"embeddings": shard}, f)

# 6) Export upper-layer entry points
for l in range(1, max_level + 1):
    entries = []
    for i in range(num_elements):
        if levels[i] >= l:
            entries.append({
                "id": i,
                "vector": data[i].tolist()
            })
    with open(f"layer{l}_entry_points.json", "w") as f:
        json.dump({"embeddings": entries}, f)

print("✅ Export complete:")
print(f"  • Layer 0: {num_shards} shards of ≤{shard_size} nodes")
print(f"  • Layers 1–{max_level}: entry point files")


✅ Export complete:
  • Layer 0: 100 shards of ≤100 nodes
  • Layers 1–3: entry point files


In [ ]:
import faiss
import numpy as np
import json
from faiss import vector_to_array
from sklearn.decomposition import PCA

# ── 1) Build your data and HNSW index ────────────────────────────────────────
num_elements = 10000
dim = 512
data = np.random.randn(num_elements, dim).astype(np.float32)
M = 16
ef_construction = 200

index = faiss.IndexHNSWFlat(dim, M)
index.hnsw.efConstruction = ef_construction
index.add(data)
hnsw = index.hnsw

# ── 2) Extract Layer 0 neighbor lists ───────────────────────────────────────
M0 = hnsw.nb_neighbors(0)
flat_nbrs = vector_to_array(hnsw.neighbors)  # length = num_elements * M0

base_neighbors = []
for i in range(num_elements):
    block = flat_nbrs[i*M0:(i+1)*M0]
    # filter out empty slots
    nbrs = [int(x) for x in block if x != -1]
    base_neighbors.append(nbrs)

# ── 3) Get node levels (for entry-point selection) ───────────────────────────
levels = vector_to_array(hnsw.levels).astype(int)   # one level per node
max_level = int(hnsw.max_level)

# ── 4) Partition & quantize the base layer ─────────────────────────────────
shard_size = 100
n_shards = (num_elements + shard_size - 1) // shard_size

for shard_id in range(n_shards):
    start = shard_id * shard_size
    end = min(start + shard_size, num_elements)
    ids = np.arange(start, end)
    vs = data[ids]                                 # (<=100, 512)
    nbrs = [base_neighbors[i] for i in ids]

    # quantize this shard to Int8 with one global scale
    max_abs = np.max(np.abs(vs))
    scale = max_abs / 127.0
    qvs = np.round(vs / scale).astype(np.int8)     # (<=100, 512)

    out = {
      "scale": float(scale),
      "embeddings": []
    }
    for idx, qv, nb in zip(ids, qvs, nbrs):
        out["embeddings"].append({
          "id": int(idx),
          "qv": qv.tolist(),       # Int8 array
          "neighbors": nb          # small list of ints
        })

    with open(f"layer0_shard_{shard_id}.json","w") as f:
        json.dump(out, f)

# ── 5) Build & quantize Layer 1 entry points via PCA → 64 dims ──────────────
#    We take all nodes with level ≥1 as “entry points”
entry_ids = [i for i, lvl in enumerate(levels) if lvl >= 1]
entry_vecs = data[entry_ids]

# PCA down to 64 dims
pca = PCA(n_components=64)
lowd = pca.fit_transform(entry_vecs)             # (n_entries, 64)

# quantize entire set with one global scale
max_abs_e = np.max(np.abs(lowd))
scale_e = max_abs_e / 127.0
q_e = np.round(lowd / scale_e).astype(np.int8)   # (n_entries, 64)

layer1 = {
  "scale": float(scale_e),
  "entries": []
}
for idx, qv in zip(entry_ids, q_e):
    layer1["entries"].append({
      "id": int(idx),
      "qv": qv.tolist()
    })

with open("layer1_entry_points.json","w") as f:
    json.dump(layer1, f)

print(f"✓ Exported {n_shards} base-layer shards (~100 nodes each, ∼200–300 KB gzipped)")
print(f"✓ Exported Layer 1 entry points (~{len(entry_ids)} nodes, ∼300 KB gzipped)")


✓ Exported 100 base-layer shards (~100 nodes each, ∼200–300 KB gzipped)
✓ Exported Layer 1 entry points (~10000 nodes, ∼300 KB gzipped)


In [ ]:
import faiss
import numpy as np
import json
from faiss import vector_to_array

# ── Configuration ───────────────────────────────────────────────
num_elements    = 10000   # total number of vectors
dim             = 512     # embedding dimension
M               = 16      # HNSW max connections per node
ef_construction = 200     # HNSW construction effort factor
shard_size      = 100     # items per shard

# ── 1) Load or generate your embeddings ──────────────────────────
# Replace the following with your actual data loading:
# data = np.load("your_embeddings.npy")  # shape (num_elements, dim)
data = np.random.randn(num_elements, dim).astype(np.float32)

# ── 2) Build the HNSW index with Faiss ─────────────────────────
index = faiss.IndexHNSWFlat(dim, M)
index.hnsw.efConstruction = ef_construction
index.add(data)
hnsw = index.hnsw

# ── 3) Extract base-layer (Layer 0) neighbor lists ───────────────
M0 = hnsw.nb_neighbors(0)                 # slots per node in layer 0
flat_nbrs = vector_to_array(hnsw.neighbors)  # length = num_elements * M0

base_neighbors = []
for i in range(num_elements):
    start = i * M0
    block = flat_nbrs[start:start + M0]
    # filter out empty slots (-1)
    nbrs = [int(x) for x in block if x != -1]
    base_neighbors.append(nbrs)

# ── 4) Extract node levels (highest layer each node appears in) ──
levels    = vector_to_array(hnsw.levels).astype(int)  # one level per node
max_level = int(hnsw.max_level)

# ── 5) Export each layer with sharding & Int8 quantization ───────
for layer in range(max_level + 1):
    # determine the node IDs belonging to this layer
    if layer == 0:
        node_ids = np.arange(num_elements)
    else:
        node_ids = np.where(levels == layer)[0]

    n_nodes  = len(node_ids)
    n_shards = (n_nodes + shard_size - 1) // shard_size

    for shard_id in range(n_shards):
        start = shard_id * shard_size
        end   = min(start + shard_size, n_nodes)
        ids   = node_ids[start:end]
        vs    = data[ids]                   # shape (≤shard_size, dim)

        # Int8 quantization with a single global scale per shard
        max_abs = np.max(np.abs(vs))
        scale   = max_abs / 127.0
        qvs     = np.round(vs / scale).astype(np.int8)

        out = {
            "scale": float(scale),
            "nodes": []
        }

        for idx, qv in zip(ids, qvs):
            node_entry = {
                "id": int(idx),
                "qv":   qv.tolist()
            }
            # include neighbor lists only in the base layer
            if layer == 0:
                node_entry["neighbors"] = base_neighbors[idx]
            out["nodes"].append(node_entry)

        filename = f"layer{layer}_shard_{shard_id}.json"
        with open(filename, "w") as f:
            json.dump(out, f, separators=(",",":"))

    print(f"Layer {layer}: exported {n_shards} shard files")

print("All layers exported successfully.")


Layer 0: exported 100 shard files
Layer 1: exported 94 shard files
Layer 2: exported 6 shard files
Layer 3: exported 1 shard files
All layers exported successfully.


In [ ]:
# !rm -rf layer3_*.json

In [ ]:
# assume these were produced during export:
#   levels:  np.array(shape=(num_elements,), dtype=int)
#   shard_size: int
#   max_level: int

# Precompute the sorted node lists per layer
layer_node_ids = {}
layer_node_ids[0] = np.arange(num_elements)
for layer in range(1, max_level+1):
    layer_node_ids[layer] = np.where(levels == layer)[0]

def shard_for(node_id: int, layer: int) -> str:
    """
    Return the shard filename in which `node_id` lives at `layer`.
    """
    ids = layer_node_ids[layer]
    # find its position
    # for speed in Python you might build a dict mapping id->pos
    pos = int(np.nonzero(ids == node_id)[0][0])
    shard_id = pos // shard_size
    return f"layer{layer}_shard_{shard_id}.json"

# Usage examples:
print(shard_for(42, 0))    # e.g. "layer0_shard_0.json"
print(shard_for(1234, 1))  # e.g. "layer1_shard_12.json"


layer0_shard_0.json
layer1_shard_11.json


In [ ]:
lookup = {}
for layer in range(max_level+1):
    lookup[layer] = {}
    ids = layer_node_ids[layer]
    for pos, nid in enumerate(ids):
        lookup[layer][int(nid)] = pos // shard_size

# dump to JSON
with open("layer_shard_lookup.json","w") as f:
    json.dump(lookup, f)


In [ ]:
layer_node_ids[3].shape

(20,)

## Quantizing the embedding model

If your selected embedding model has no quantized ONNX version yet, you can perform the quantization process using the steps below.

You will need to publish the ONNX versions of the model in the `./onnx/` directory in a huggingface models repo. Ensure to upload the `model.onnx`, `model_quantized.onnx`, and `ort_config.json` files.

In [ ]:
!pip install optimum[exporters] &> /dev/null

In [ ]:
%%bash

optimum-cli export onnx -m avsolatorio/GIST-small-Embedding-v0 GIST_small_Embedding_v0_onnx/

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
2025-04-17 04:16:45.819647: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744863405.841565    6250 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744863405.849621    6250 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [ ]:
%%bash

optimum-cli onnxruntime quantize \
  --avx512 \
  --onnx_model GIST_small_Embedding_v0_onnx \
  -o GIST_small_Embedding_v0_onnx_quantized

2025-04-17 04:17:46.248430: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744863466.280563    6532 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744863466.290748    6532 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
